<a href="https://colab.research.google.com/github/alkadevirajbanshi-ai/Intelligence-System/blob/main/ResearchAssistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Google Gen AI SDK + utilities
!pip install --upgrade google-genai arxiv beautifulsoup4 requests readability-lxml

In [12]:
from getpass import getpass
import os
api_key = getpass("Paste your Google Gemini API key here: ")
os.environ["GEMINI_API_KEY"] = api_key # optional - keep in memory only

Paste your Google Gemini API key here: ··········


In [13]:
from google import genai
# create client using the key we provided
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [22]:
resp = client.models.generate_content(
 model="gemini-3-flash-preview",
 contents="Summarize in one sentence: Why is reproducibility important in research?"
)
print(resp.text)

Reproducibility is essential because it validates the reliability of scientific findings, ensuring that results are credible, verifiable, and provide a solid foundation for future research.


In [23]:
import arxiv

def search_arxiv(query, max_results=5):
    # Initialize the client
    client = arxiv.Client()

    # Define the search
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    results = []
    # Use client.results() to fetch results
    for result in client.results(search):
        results.append({
            "title": result.title,
            "summary": result.summary,
            "authors": [a.name for a in result.authors],
            "pdf_url": result.pdf_url,
            "id": result.entry_id.split('/abs/')[-1]  # Get short ID from entry_id URL
        })
    return results

# quick test
papers = search_arxiv("multimodal transformers", max_results=3)
for p in papers:
    print(p['title'])

Introducing Representations of Facial Affect in Automated Multimodal Deception Detection
Low Rank Fusion based Transformers for Multimodal Sequences
On The Application of Linear Attention in Multimodal Transformers


In [24]:
import requests
from bs4 import BeautifulSoup
def fetch_page_text(url, max_chars=4000):
 try:
  r = requests.get(url, timeout=10, headers={"User-Agent":"research-agent/1.0"})
  r.raise_for_status()
 except Exception as e:
  return ""
 soup = BeautifulSoup(r.text, "html.parser")
 # simple extraction: join visible <p> text
 paragraphs = [p.get_text(separator=" ", strip=True) for p in soup.find_all("p")]
 content = "\n\n".join(paragraphs)
 return content[:max_chars] # limit length for API

In [25]:
def chunk_text(text, max_chars=3000):
 chunks = []
 start = 0
 while start < len(text):
  end = min(len(text), start + max_chars)
  chunks.append(text[start:end])
  start = end
 return chunks

In [26]:
SYSTEM_PROMPT = (
 "You are a concise research assistant. For each text provided, return: "
 "1) a one-paragraph summary (3-5 sentences), 2) three bullet key contributions/findings, "
 "and 3) a suggested short title. Be factual and include no hallucinated facts."
)
def summarize_chunk(chunk_text):
 prompt = SYSTEM_PROMPT + "\n\nText to summarize:\n" + chunk_text
 resp = client.models.generate_content(
 model="gemini-3.5-flash", # adjust if unavailable
 contents=prompt
 )
 return resp.text

In [27]:
def summarize_text_long(text):
 chunks = chunk_text(text, max_chars=3000)
 summaries = []
 for c in chunks:
  s = summarize_chunk(c)
  summaries.append(s)
  # Optionally aggregate the chunk summaries into a single final summary:
  if len(summaries) == 1:
   return summaries[0]
  else:
    combined = "\n\n".join(summaries)
    final_prompt = SYSTEM_PROMPT + "\n\nCombine the following chunk summaries into a single concise summary:\n\n" + combined
    final_resp = client.models.generate_content(model="gemini-3.5-flash", contents=final_prompt)
 return final_resp

In [28]:
from tenacity import retry, stop_after_attempt, wait_exponential

# We define a robust helper function that retries if the Gemini API is temporarily unavailable
@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=2, max=10))
def robust_summarize(text):
    return summarize_text_long(text)

query = "multimodal transformers"
papers = search_arxiv(query, max_results=5)
for p in papers:
    print("\n---")
    print("Title:", p['title'])
    text_to_summarize = p['summary'] # arXiv abstract (short)
    try:
        summary = robust_summarize(text_to_summarize)
        print("Summary:\n", summary)
    except Exception as e:
        print(f"Could not summarize paper due to temporary API error: {e}")
    print("PDF:", p['pdf_url'])


---
Title: Introducing Representations of Facial Affect in Automated Multimodal Deception Detection
Summary:
 **Suggested Short Title:**  
Affect-Aware Multimodal Deception Detection Using Facial Affect

**Summary:**  
This paper explores automated deception detection in high-stakes courtroom situations by analyzing continuous dimensional representations of facial affect (valence and arousal) alongside visual, vocal, and verbal features. Using a deep neural network trained on the Aff-Wild database to extract facial affect, the authors evaluated unimodal Support Vector Machines (SVM) and multimodal fusion models. The unimodal facial affect model achieved an AUC of 80%, while an adaptive boosting multimodal model reached an AUC of 91% on unseen speakers, outperforming existing approaches that lacked facial affect features. Ultimately, the significant differences found in facial valence and arousal between deceptive and truthful speakers provide computational support for psychological th

In [29]:
from google.colab import drive
from tenacity import retry, stop_after_attempt, wait_exponential

# Mount Google Drive
drive.mount('/content/drive')
out_path = '/content/drive/MyDrive/research_summaries.txt'

# Set up a robust summarization helper
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=6), reraise=True)
def robust_summarize(text):
    return summarize_text_long(text)

with open(out_path, 'w', encoding='utf-8') as f:
    for p in papers:
        f.write("TITLE: " + p['title'] + "\n")
        try:
            print(f"Processing: {p['title']}...")
            s = robust_summarize(p['summary'])
            f.write(s + "\n\n---\n")
        except Exception as e:
            print(f"Skipping summary for '{p['title']}' due to API limit or error: {e}")
            f.write("Summary unavailable (API Quota Exhausted/Error).\n\n---\n")

print("Saved to", out_path)

Mounted at /content/drive
Processing: Introducing Representations of Facial Affect in Automated Multimodal Deception Detection...
Skipping summary for 'Introducing Representations of Facial Affect in Automated Multimodal Deception Detection' due to API limit or error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing: Low Rank Fusion based Transformers for Multimodal Sequences...
Processing: On The Application of Linear Attention in Multimodal Transformers...
Processing: MM-VeriRec: Failure-Guided Fusion for Verifiable Agentic Multimodal Recommendation...
Skipping summary for 'MM-VeriRec: Failure-Guided Fusion for Verifiable Agentic Multimodal Recommendation' due to API limit or error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please